# Notebook 40 — Reproducible LLM Benchmarking with LightEval

    ## Learning objectives

    - Distinguish likelihood, generation, and judge tasks
- Define reproducible benchmark configurations
- Interpret aggregate scores, contamination, and uncertainty

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['lighteval>=0.12']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 40.1 A harness standardizes mechanics

A benchmark harness makes prompts, few-shot selection, batching, metrics, model adapters, and result serialization repeatable. It does not make a benchmark representative or uncontaminated. LightEval is the actively maintained Hugging Face-aligned option for LLM evaluation. Pin its version, task revision, model/tokenizer/template, generation configuration, and backend. Run a small smoke subset before expensive evaluation and preserve per-example outputs.


In [ ]:
config={"harness":"lighteval@pinned","tasks":["task@revision"],"fewshot":0,"seed":42,"model":"org/model@commit","template":"tokenizer@commit"}; print(config)


## 40.2 Task types

Log-likelihood tasks compare candidate continuation probabilities and require correct token boundaries and normalization. Generative tasks decode an answer and apply exact, normalized, executable, semantic, or judge graders. Few-shot examples alter context and must be fixed by seed and selection policy. Chat templates can change results materially. Stop sequences, maximum tokens, sampling, and answer extraction are part of the benchmark definition, not incidental CLI flags.


In [ ]:
items=[{"type":"likelihood","scores":[-.2,-1.4],"gold":0},{"type":"generation","output":"42","reference":"42"}]; print(items)


## 40.3 Statistics and contamination

Aggregate accuracy conceals slices and item dependence. Report counts, paired differences, bootstrap intervals where appropriate, and failures. Pass@k depends on number and sampling distribution. Benchmark data may appear in pretraining or synthetic instruction corpora; use contamination checks, private or newly created tests, temporal splits, and perturbations. Leaderboard comparisons are invalid when prompts, few-shot counts, revisions, or inference settings differ.


In [ ]:
import random
diffs=[1,0,-1,1,1,0]; rng=random.Random(42); means=[sum(rng.choices(diffs,k=len(diffs)))/len(diffs) for _ in range(1000)]; print(sorted(means)[25],sorted(means)[975])


## 40.4 Operational workflow

Create a manifest, inspect rendered prompts, run deterministic baselines, execute the candidate and parent under identical settings, validate completion and error counts, then publish raw predictions plus summary. Distributed evaluation must avoid duplicate items and aggregate numerators/denominators correctly. Quality changes after quantization or serving-engine migration require the same frozen suite. Standard benchmarks complement product-specific tasks, safety tests, RAG and agent trajectories; none replaces them.


In [ ]:
results={"completed":98,"errors":2,"expected":100}; assert results["completed"]+results["errors"]==results["expected"]; print(results)


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 40.5 Inspect harness-rendered samples

A benchmark name is insufficient. Inspect the exact document-to-prompt mapping, few-shot examples, chat template, answer choices, continuation boundary, stop strings, generation settings, and metric normalization. Run a handful of items through a transparent manual implementation and compare with harness outputs. Pin LightEval, task revisions, model/tokenizer commits, and backend. Preserve raw predictions and request errors; a completed aggregate with silently skipped items is invalid.


In [ ]:
sample={"task":"custom@v1","doc_id":"7","rendered":"Question: 2+2\nAnswer:","gold":"4","fewshot_ids":[],"stop":["\n"],"seed":42}; print(sample)


## 40.6 Comparable benchmark runs

Compare checkpoints with the same prompt, template, few-shot count, seed policy, dtype, quantization, backend, and generation settings. Report item counts and paired deltas with uncertainty. For generative metrics, audit extraction failures separately from wrong answers. Distributed execution must avoid duplicate examples and aggregate sums/counts correctly. Standard leaderboards complement private, recent, adversarial, product, RAG, and agent suites; they do not replace them.


In [ ]:
run_a={"correct":[1,0,1,1,0]}; run_b={"correct":[1,1,1,0,1]}; diff=[b-a for a,b in zip(run_a["correct"],run_b["correct"])]; print("paired delta",sum(diff)/len(diff),"wins",diff.count(1),"losses",diff.count(-1))


## 40.7 A real LightEval workflow

Use the installed LightEval CLI to list tasks and inspect a task before running it: `lighteval tasks list` and `lighteval tasks inspect <task>`. A Python run composes an `EvaluationTracker`, backend-specific model configuration, `PipelineParameters`, and `Pipeline`; custom tasks are supplied through an explicit directory. Choose a stable package release unless the course deliberately pins the development branch. Begin with a tiny task subset, keep the output bundle local, and inspect per-sample records before publishing. Backend names and constructor signatures change, so the notebook introspects the installed version rather than presenting an unverified executable training-scale command. A production manifest must still pin task, model, tokenizer/template, backend, dtype, few-shot selection, generation, and code revisions.


In [ ]:
import importlib.util,subprocess
commands=[["lighteval","tasks","list"],["lighteval","tasks","inspect","truthfulqa:mc"]]
print("inspection commands:"); print(*(" ".join(c) for c in commands),sep="\n")
if importlib.util.find_spec("lighteval"):
 import lighteval
 print("installed LightEval",getattr(lighteval,"__version__","unknown"))
 try:
  from lighteval.pipeline import Pipeline,PipelineParameters,ParallelismManager
  from lighteval.logging.evaluation_tracker import EvaluationTracker
  print("pipeline components available",Pipeline,PipelineParameters,ParallelismManager,EvaluationTracker)
 except ImportError as exc: print("Installed-version API differs; consult its pinned docs:",exc)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [LightEval documentation](https://huggingface.co/docs/lighteval/index)
- [LightEval Python API](https://huggingface.co/docs/lighteval/using-the-python-api)
- [LightEval task inspection](https://huggingface.co/docs/lighteval/available-tasks)


## Exercises

    1. Define a custom task.
2. Compare two checkpoints with paired intervals.
3. Audit template and contamination sensitivity.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
